# BM25 / Ruri / BGE-M3 / Qwen 組み合わせ RAG ベースライン

**目的**: 日本語 Wikipedia のチャンク（約100万件）から根拠を検索し、短答式の質問301問に答える。この notebook は配布済みインデックスを使い、正解を含まない質問ファイルで推論して提出ファイルを作る。採点は行わない。

| 項目 | 内容 |
|---|---|
| 質問数 | 301問 |
| チャンク | 1,003,139件（日本語 Wikipedia 49,043記事を約500文字ごとに分割） |
| 実行環境 | Google Colab（GPU: T4、ランタイムのバージョン 2026.07） |

**回答の形式**: 生成モデルは、回答と、その根拠にしたチャンクの ID を1つだけ出力する。

```json
{"answer": "1803年9月28日", "chunk_id": 500276}
```

**提出形式**: `submission.json` に、`q_id`・`answer`・`chunk` の3項目だけを持つ JSON 配列を保存する。評価は別の運営用スクリプトで行う。

**ベースラインの構成**: Hugging Face のデータセットから取得したインデックスをそのまま使うので、インデックスの作成は不要である。検索は BM25・Ruri・BGE-M3・Qwen から任意の組み合わせを選べる。デフォルトは Ruri 単体・リランカーなし。

| 工程 | 設定 |
|---|---|
| BM25 | SudachiPy（SplitMode C、辞書形）で分かち書きし、Okapi BM25（k1 = 1.2、b = 0.75）で上位100件 |
| 埋め込み検索 | Ruri / BGE-M3 / Qwen のうち選択した各モデルで上位100件 |
| 統合 | 検索を2つ以上選んだ場合だけ、各検索の順位を等重みの RRF（k = 60）で統合して上位100件。1つなら順位をそのまま使う |
| リランク | 使う場合（`USE_RERANKER = True`）は、`RERANKER_NAME` で選んだリランカー（Ruri / BGE / Qwen。デフォルトは Ruri）で統合結果の上位50件を並べ替え、上位10件を使う。 |
| 回答生成 | `Qwen/Qwen3.5-0.8B` を vLLM で実行。サンプリングはモデルカードの推奨値、thinking 無効、最大1024トークン、出力は JSON スキーマで制約 |

**事前に済ませておくこと**:
- Hugging Face のアカウントを作り、データセット [`bumbuboon/llm-course-final-2026`](https://huggingface.co/datasets/bumbuboon/llm-course-final-2026) のページで利用条件に同意する
- Hugging Face の[トークン設定ページ](https://huggingface.co/settings/tokens)で Read 権限のトークンを作り、Colab の左の鍵アイコン（シークレット）に `HF_READ_TOKEN` という名前で追加して「ノートブックからのアクセス」を有効にする。データとインデックスの取得に使う。書き込み権限は不要
- Colab のメニュー「ランタイム」→「ランタイムのタイプを変更」で、ハードウェア アクセラレータに **T4 GPU** を選び、ライブラリの版を固定するためにランタイムのバージョンに **2026.07** を選ぶ。最新版を選ぶと、Colab の更新でライブラリが変わり動かなくなることがある

## データの置き場所

質問は、マイドライブの `LLM講座最終コンペ_work/questions.json`（設定セルの `WORK_DIR` の下）から読み込む。JSON 配列の各要素は `{"q_id": "q001", "question": "質問文"}` の2項目だけで、正解や根拠 ID は含まない。

Drive にこのファイルがない場合だけ、Hugging Face の `data/questions.json` を取得して同じ場所に保存する。ファイルがすでにあればそのまま使い、上書きしない。

チャンクとインデックスは、Hugging Face から Colab のローカルディスク（`/content/rag_data`）に取得する。

実行結果は `LLM講座最終コンペ_work/<RUN_ID>/` に保存する。提出ファイル `submission.json` のほか、検索設定の `retrieval_config.json` と生成設定の `generation_config.json` を置く。

## 利用ルール

### 使用できる深層学習モデル

エンコーダ・リランカー・LLM のいずれも、以下のリストにあるモデルだけを使用できる。

| 用途 | モデルID |
|---|---|
| エンコーダ | `cl-nagoya/ruri-v3-310m` |
| エンコーダ | `BAAI/bge-m3` |
| エンコーダ | `Qwen/Qwen3-Embedding-0.6B` |
| リランカー | `cl-nagoya/ruri-v3-reranker-310m` |
| リランカー | `BAAI/bge-reranker-v2-m3` |
| リランカー | `Qwen/Qwen3-Reranker-0.6B` |
| LLM | `Qwen/Qwen3.5-0.8B` |

### 禁止事項

- 外部 API の使用は一切禁止する。
- 配布チャンクの本文や分割を変更することは禁止する。
- 配布外データ（Web 検索、別のコーパス、出典の Wikipedia 記事など）を参照することは禁止する。
- リストにない深層学習モデルを使用することは禁止する。エンコーダ・リランカー・LLM のすべてが対象である。
- 指定されたもの以外のエンコーダで埋め込みを作成することは禁止する。
  - ただし、指定されたエンコーダを学習した派生モデルは使用でき、それで埋め込みを作成してよい。
- `Qwen/Qwen3.5-0.8B` 以外の LLM で、質問文やチャンクの加工・アノテーションを行うことは禁止する。
- `Qwen/Qwen3.5-0.8B` 以外の LLM による推論・生成は禁止する。
- 深層学習以外の機械学習モデルも、Colab 以外で実行することは禁止する。

### 許可事項

以下は、上記のモデル・データ・実行環境の制約の範囲で許可する。

- 検索パラメータの変更。
- 指定された範囲内でのエンコーダの切り替え。
- ハイブリッド検索（複数の検索方式の組み合わせ）。
- 指定されたエンコーダの学習。
- 指定されたリランカーの学習。
- マルチクエリ。
- エージェンティックサーチ。
- 推論パラメータの変更。
- プロンプトの調整。
- 深層学習以外の機械学習モデルを Colab 上で実行すること。

マルチクエリやエージェンティックサーチで LLM を使う場合も、使用できる LLM は `Qwen/Qwen3.5-0.8B` だけである。

## Step 0: 環境セットアップ

vLLM（LLM 推論）と SudachiPy（分かち書き）を、版を固定して入れる。

- vLLM は PyPI 版が CUDA 13 用で、このランタイムの torch（CUDA 12 系）と組み合わせられないため、GitHub から入れる。
- vLLM が torch を入れ替えることを防ぐため、torch を固定する。
- torchcodec が vLLM の読み込みを妨げるため外す。
- 依存関係の警告（numba、protobuf など）が出るが、Colab に最初から入っている別のパッケージに関するもので、この notebook の処理には影響しない。

In [ ]:

!nvidia-smi

Sat Nov 14 17:44:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# [step0.1 setup-install]  ←  このセルの名前（以降のセルも1行目に名前を記載）
VLLM_WHEEL = "https://github.com/vllm-project/vllm/releases/download/v0.26.0/vllm-0.26.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"
!printf "torch==2.11.0\ntorchvision==0.26.0\n" > /tmp/constraints.txt
!pip install -q "$VLLM_WHEEL" "sudachipy==0.6.11" "sudachidict-core==20260428" -c /tmp/constraints.txt > /tmp/pip_install.log 2>&1
print("exit code:", _exit_code)   # 直前の pip install の終了コード（0 なら成功）
!pip uninstall -y -q torchcodec
!tail -3 /tmp/pip_install.log

exit code: 0
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.


**⚠ 上のセルの実行後、「ランタイム」→「セッションを再起動する」を実施し、下の [step1.1 config] 以降を実行する。**

vLLM は、インストール前に読み込まれた torch などのモジュールが残っていると正しく初期化できないため。

## Step 1: BM25 / Ruri / BGE-M3 / Qwen を自由に組み合わせる

設定セルの `RETRIEVERS` に、使いたい検索だけを並べる。空でない15通りの組み合わせに対応する。

```python
RETRIEVERS = ["qwen"]                        # 単体
RETRIEVERS = ["ruri", "bge"]                # 埋め込み同士
RETRIEVERS = ["bm25", "bge", "qwen"]        # 3種類
RETRIEVERS = ["bm25", "ruri", "bge", "qwen"] # 全部
```

各検索の候補数は `BM25_TOP_K`・`RURI_TOP_K`・`BGE_TOP_K`・`QWEN_TOP_K` で個別に指定する。検索が2つ以上なら等重みの RRF で統合し、1つなら順位をそのまま使う。統合後の候補数は `FUSED_TOP_K` である。リランカーを使うかどうかは `USE_RERANKER` で、使うリランカーは `RERANKER_NAME`（`"ruri"`・`"bge"`・`"qwen"`）で選ぶ。デフォルトは `RETRIEVERS = ["ruri"]`、`USE_RERANKER = False`（Ruri 単体・リランカーなし）である。

インデックスは、Hugging Face のデータセットの `index/` の下にある次のフォルダから取得する。BM25 は `bm25-sudachi-c/`、Ruri は `ruri-v3-310m/`、BGE-M3 は `bge-m3/`、Qwen は `qwen3-embedding-0.6b/` である。

質問の埋め込み設定は `EMBED_CONFIGS` で指定する。デフォルト値は配布インデックスの `index_config.json` と同じである。model・revision・pooling・normalize・trust_remote_code は文書ベクトルを作ったときと揃える必要がある。異なる場合は検索精度が下がる。文書ベクトルを別の設定で作り直した場合は、`EMB_INDICES` をそのインデックスに変え、`EMBED_CONFIGS` も合わせる。`query_prefix`・`max_length` は質問側だけの設定なので、変更可能である。

追加学習したモデルなどで独自の埋め込み検索を加える場合は、新しい名前を `RETRIEVERS` に入れ、同じ名前で `EMB_INDICES`・`EMBED_CONFIGS`・`RETRIEVER_TOP_K` に設定を加える。`RETRIEVERS` が空の場合、同じ名前が重複している場合、設定が足りない場合は、設定セルがエラーで止まる。

出力フォルダ名は `RUN_ID` で指定する。条件を変えて結果を残す場合は `RUN_ID` も変える。設定を変えたときは、ランタイムを再起動して設定セルから実行し直す。

In [ ]:
sdffadsfas

NameError: name 'sdffadsfas' is not defined

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# # 【キャッシュ作成セル】1回だけ。データとモデルを、Drive に1つずつの tar ファイルとして保存する
# # 前提：このセッションで、検索（bm25・ruri・bge）まで実行して、/content/rag_data と ~/.cache/huggingface がそろっていること
# import os, subprocess, time
# CACHE = "/content/drive/MyDrive/LLM講座最終コンペ_cache"
# os.makedirs(CACHE, exist_ok=True)
# assert os.path.isfile("/content/rag_data/index/chunk_offsets.npy"), "/content/rag_data がそろっていない。先に [step1.3 storage] まで実行する"
# for name, parent, folder, tar in [
#     ("rag_data", "/content", "rag_data", CACHE + "/rag_data.tar"),
#     ("モデル", "/root/.cache", "huggingface", CACHE + "/hf_cache.tar"),
# ]:
#     t0 = time.time()
#     tmp = tar + ".tmp"
#     subprocess.run(["tar", "-cf", tmp, "-C", parent, folder], check=True)
#     os.replace(tmp, tar)
#     print("%s: %.1f GB / %.0f 秒" % (name, os.path.getsize(tar) / 1e9, time.time() - t0))

In [ ]:
# import time
# CACHE = "/content/drive/MyDrive/LLM講座最終コンペ_cache"
# t0 = time.time()
# !rsync -a --info=progress2 "$CACHE/rag_data/" /content/rag_data/
# print("rag_data 復元: %.0f 秒" % (time.time() - t0))

In [ ]:
# [step0.9 cache-restore] Drive のキャッシュから、データとモデルを Colab のローカルディスクに戻す
# tar があれば tar から（速い）、なければ元のフォルダから rsync で戻す。復元済みなら飛ばす
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, time
CACHE = "/content/drive/MyDrive/LLM講座最終コンペ_cache"
JOBS = [
    ("rag_data", CACHE + "/rag_data.tar", "/content", CACHE + "/rag_data/", "/content/rag_data/", "/content/.restored_rag_data"),
    ("モデル", CACHE + "/hf_cache.tar", "/root/.cache", CACHE + "/hf_cache/", "/root/.cache/huggingface/", "/content/.restored_hf_cache"),
]
t0 = time.time()
procs = []
for name, tar, tar_dest, folder, folder_dest, mark in JOBS:
    if os.path.exists(mark):
        print(name, ": 復元済みなので飛ばす")
        continue
    if os.path.exists(tar):
        os.makedirs(tar_dest, exist_ok=True)
        cmd, how = ["tar", "-xf", tar, "-C", tar_dest], "tar"
        print(name, ": tar %.1f GB" % (os.path.getsize(tar) / 1e9))
    elif os.path.isdir(folder):
        os.makedirs(folder_dest, exist_ok=True)
        cmd, how = ["rsync", "-a", folder, folder_dest], "rsync"
    else:
        print(name, ": キャッシュがないので飛ばす（[step1.3 storage] などで Hugging Face から取得する）")
        continue
    print(name, ":", how, "で復元を始めた")
    procs.append((name, how, mark, subprocess.Popen(cmd)))
def _stop_all(msg):
    for _, _, _, q in procs:      # 失敗したときは、残りの復元も止める（裏で動き続けないように）
        if q.poll() is None:
            q.terminate()
    raise SystemExit(msg)
for name, how, mark, p in procs:
    if p.wait() != 0:
        _stop_all(name + " の復元（" + how + "）に失敗した")
    if name == "rag_data" and not os.path.isfile("/content/rag_data/index/chunk_offsets.npy"):
        _stop_all("rag_data を戻したのに index/chunk_offsets.npy がない。tar が古いか壊れている。下のキャッシュ作成セルで作り直す")
    open(mark, "w").close()
    print(name, ": 復元した（%s・開始から %.0f 秒）" % (how, time.time() - t0))
print("復元: %.0f 秒" % (time.time() - t0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
rag_data : tar 7.1 GB
rag_data : tar で復元を始めた
モデル : tar 8.9 GB
モデル : tar で復元を始めた
rag_data : 復元した（tar・開始から 167 秒）
モデル : 復元した（tar・開始から 197 秒）
復元: 197 秒


In [ ]:
# [step1.1 config]
# =====================================================================
# 1. 検索の組み合わせと実行名
# =====================================================================
# ここだけ変更すれば単体・任意の組み合わせで実行できる。
RETRIEVERS = ["bm25", "ruri", "bge"]            # 選択肢: "bm25", "ruri", "bge", "qwen"（独自の埋め込み検索を加えた場合はその名前）
# 例: ["qwen"] / ["ruri", "bge"] / ["bm25", "ruri", "bge", "qwen"]
RUN_ID = "dev_ctx_dump"

# =====================================================================
# 2. インデックス
# =====================================================================
BM25_INDEX = "bm25-sudachi-c"     # index/ 直下のフォルダ名
EMB_INDICES = {"ruri": "ruri-v3-310m", "bge": "bge-m3", "qwen": "qwen3-embedding-0.6b"}

# =====================================================================
# 3. モデル
# =====================================================================
# 質問の埋め込みに使う設定。デフォルト値は配布インデックスの index_config.json と同じ。
# 注意: model・revision・pooling・normalize・trust_remote_code は、文書ベクトルを作ったときと同じにすること。
#       違ってもエラーにはならず、検索精度だけが大きく下がる。
#       文書ベクトルを別の設定で作り直した場合は、EMB_INDICES をそのインデックスに変え、ここも合わせる。
#       query_prefix・max_length は質問側だけの設定なので、変えて試せる。
# index_config.json と食い違う項目は [step3.2 embedding] で表示する。
EMBED_CONFIGS = {
    "ruri": {
        "model": "cl-nagoya/ruri-v3-310m",
        "revision": "18b60fb8c2b9df296fb4212bb7d23ef94e579cd3",
        "query_prefix": "検索クエリ: ",
        "pooling": "mean",            # "mean" | "cls" | "last"
        "normalize": True,            # True: L2 正規化（内積 = コサイン類似度）
        "max_length": 8192,
        "trust_remote_code": False,
    },
    "bge": {
        "model": "BAAI/bge-m3",
        "revision": "5617a9f61b028005a4858fdac845db406aefb181",
        "query_prefix": "",
        "pooling": "cls",
        "normalize": True,
        "max_length": 8192,
        "trust_remote_code": False,
    },
    "qwen": {
        "model": "Qwen/Qwen3-Embedding-0.6B",
        "revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",
        "query_prefix": "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: ",
        "pooling": "last",
        "normalize": True,
        "max_length": 8192,
        "trust_remote_code": False,
    },
}
EMBED_BATCH_SIZE = 1              # 質問埋め込みのバッチサイズ（T4の長い質問にも配慮）
DENSE_QUERY_BATCH_SIZE = 32       # 行列検索時にまとめて処理する質問数

# リランカーの設定。kind は計算方法: "cross_encoder"（Ruri・BGE）| "qwen3"（Qwen3-Reranker の yes / no 判定）
RERANKER_CONFIGS = {
    "ruri": {
        "model": "cl-nagoya/ruri-v3-reranker-310m",
        "revision": "2c23c61a90583f0348880a2ffb2d7fd29e2d14f1",
        "kind": "cross_encoder",
        "max_length": 512,
        "batch_size": 4,
    },
    "bge": {
        "model": "BAAI/bge-reranker-v2-m3",
        "revision": "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e",
        "kind": "cross_encoder",
        "max_length": 512,
        "batch_size": 4,
    },
    "qwen": {
        "model": "Qwen/Qwen3-Reranker-0.6B",
        "revision": "e61197ed45024b0ed8a2d74b80b4d909f1255473",
        "kind": "qwen3",
        "instruction": "Given a web search query, retrieve relevant passages that answer the query",
        "max_length": 1024,           # 指示文と書式の分だけ長くなるため、Ruri・BGE より長くする
        "batch_size": 4,
    },
}
USE_RERANKER = True              # False: 統合結果の上位をそのまま使う
RERANKER_NAME = "ruri"            # USE_RERANKER = True のときに使うリランカー: "ruri" | "bge" | "qwen"

DEVICE = "auto"                   # 検索側のモデルの置き場所: "auto" | "cuda" | "cpu"

# =====================================================================
# 4. 検索と生成のパラメータ
# =====================================================================
FUSION = "rrf"                    # 複数の検索結果の統合方法（Step 5 の fuse を参照）
BM25_K1 = 1.2
BM25_B = 0.75
BM25_TOP_K = 100                  # BM25 の候補数
RURI_TOP_K = 100                  # Ruri の候補数
BGE_TOP_K = 100                   # BGE-M3 の候補数
QWEN_TOP_K = 100                  # Qwen の候補数
# 検索名ごとの候補数。独自の埋め込み検索を加えた場合は、ここにも同じ名前で候補数を加える
RETRIEVER_TOP_K = {"bm25": BM25_TOP_K, "ruri": RURI_TOP_K, "bge": BGE_TOP_K, "qwen": QWEN_TOP_K}
RRF_K = 60
FUSED_TOP_K = 100                # 統合後に残す件数
RERANK_CANDIDATES = 100          # リランクにかける件数（統合結果の上位から）
CONTEXT_TOP_K = 10                # 生成モデルに渡すチャンク数

# =====================================================================
# 5. データの取得元・結果の保存先と実行範囲
# =====================================================================
HF_REPO = "bumbuboon/llm-course-final-2026"   # データとインデックスを置いた Hugging Face のデータセット
HF_REVISION = "main"              # データセットのブランチ
LOCAL_DATA = "/content/rag_data"  # データとインデックスの取得先（Colab のローカルディスク）
DRIVE_ROOT = "/content/drive/MyDrive"
WORK_DIR = DRIVE_ROOT + "/LLM講座最終コンペ_eval/dev_work"       # 自分の作業フォルダ（Drive）
QUESTION_LIMIT = None             # 例: 10 にすると先頭10問だけ実行する（動作確認用）

# =====================================================================
# ここから下は派生値。編集不要。
# =====================================================================
CHUNKS_DIR = LOCAL_DATA + "/data/chunks"
QUESTIONS_PATH = WORK_DIR + "/questions.json"
INDEX_ROOT = LOCAL_DATA + "/index"   # 配布済みインデックスの取得先
DENSE_NAMES = [name for name in RETRIEVERS if name != "bm25"]
RUN_DIR = WORK_DIR + "/" + RUN_ID
if USE_RERANKER and RERANKER_NAME not in RERANKER_CONFIGS:
    raise ValueError("RERANKER_NAME の %r の設定が RERANKER_CONFIGS にない" % RERANKER_NAME)
RERANKER = RERANKER_CONFIGS[RERANKER_NAME] if USE_RERANKER else None

# RETRIEVERS の確認: 空・重複は禁止。埋め込み検索は、同じ名前の設定が3つの辞書にそろっている必要がある
if not RETRIEVERS:
    raise ValueError("RETRIEVERS が空である。使う検索を1つ以上指定すること")
_duplicates = sorted({name for name in RETRIEVERS if RETRIEVERS.count(name) > 1})
if _duplicates:
    raise ValueError("RETRIEVERS に同じ名前が重複している: %s" % _duplicates)
for name in RETRIEVERS:
    _tables = {"RETRIEVER_TOP_K": RETRIEVER_TOP_K} if name == "bm25" else {
        "EMB_INDICES": EMB_INDICES, "EMBED_CONFIGS": EMBED_CONFIGS, "RETRIEVER_TOP_K": RETRIEVER_TOP_K}
    _missing = [table for table, values in _tables.items() if name not in values]
    if _missing:
        raise ValueError("RETRIEVERS の %r の設定が %s にない。同じ名前で設定を加えること"
                         % (name, "・".join(_missing)))


print("RUN_ID     :", RUN_ID, "->", RUN_DIR)
print("RETRIEVERS :", RETRIEVERS, "/ FUSION:", FUSION if len(RETRIEVERS) > 1 else "なし")
print("TOP_K      :", {name: RETRIEVER_TOP_K[name] for name in RETRIEVERS})
print("BM25_INDEX :", BM25_INDEX if "bm25" in RETRIEVERS else "使わない")
print("EMB_INDICES:", {name: EMB_INDICES[name] for name in DENSE_NAMES})
print("RERANKER   :", "%s（%s）" % (RERANKER_NAME, RERANKER["model"]) if USE_RERANKER else "使わない")

RUN_ID     : dev_ctx_dump -> /content/drive/MyDrive/LLM講座最終コンペ_eval/dev_work/dev_ctx_dump
RETRIEVERS : ['bm25', 'ruri', 'bge'] / FUSION: rrf
TOP_K      : {'bm25': 100, 'ruri': 100, 'bge': 100}
BM25_INDEX : bm25-sudachi-c
EMB_INDICES: {'ruri': 'ruri-v3-310m', 'bge': 'bge-m3'}
RERANKER   : ruri（cl-nagoya/ruri-v3-reranker-310m）


In [ ]:
# [step1.2 env-check]
# ライブラリの版と GPU を確認する。想定と違う場合は警告を出す（結果は retrieval_config.json に記録する）
import importlib.metadata as _md
import platform
import torch
EXPECTED_VERSIONS = {"torch": "2.11.0", "vllm": "0.26.0"}
ENV = {"python": platform.python_version()}
for _pkg in ("torch", "transformers", "vllm", "sudachipy"):
    try:
        ENV[_pkg] = _md.version(_pkg)
    except _md.PackageNotFoundError:
        ENV[_pkg] = None
ENV["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
for _k, _v in ENV.items():
    print("  %-13s %s" % (_k, _v))
for _pkg, _want in EXPECTED_VERSIONS.items():
    if not (ENV.get(_pkg) or "").startswith(_want):
        print("[警告] %s の版が想定（%s）と違う: %s。ランタイムのバージョン 2026.07 を選び、[step0.1 setup-install] からやり直すこと"
              % (_pkg, _want, ENV.get(_pkg)))
if not ENV["gpu"]:
    print("[警告] GPU が使えない。ランタイムのタイプで T4 GPU を選ぶこと")

  python        3.12.13
  torch         2.11.0+cu128
  transformers  5.13.1
  vllm          0.26.0+cu129
  sudachipy     0.6.11
  gpu           Tesla T4


In [ ]:
# # [step1.2 env-check]
# # ライブラリの版と GPU を確認する。想定と違う場合は警告を出す（結果は retrieval_config.json に記録する）
# import importlib.metadata as _md
# import platform
# import torch
# from huggingface_hub import login
# login()   # Hugging Face


# EXPECTED_VERSIONS = {"torch": "2.11.0", "vllm": "0.26.0"}

# ENV = {"python": platform.python_version()}
# for _pkg in ("torch", "transformers", "vllm", "sudachipy"):
#     try:
#         ENV[_pkg] = _md.version(_pkg)
#     except _md.PackageNotFoundError:
#         ENV[_pkg] = None
# ENV["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
# for _k, _v in ENV.items():
#     print("  %-13s %s" % (_k, _v))
# for _pkg, _want in EXPECTED_VERSIONS.items():
#     if not (ENV.get(_pkg) or "").startswith(_want):
#         print("[警告] %s の版が想定（%s）と違う: %s。ランタイムのバージョン 2026.07 を選び、[step0.1 setup-install] からやり直すこと"
#               % (_pkg, _want, ENV.get(_pkg)))
# if not ENV["gpu"]:
#     print("[警告] GPU が使えない。ランタイムのタイプで T4 GPU を選ぶこと")

In [ ]:
# [step1.3 storage]
# 1) 配布済みデータと選択した検索インデックスをReadトークンで取得する
# 2) Drive をマウントし、作業フォルダ（実行結果の保存先）を作る
import json, os, time, getpass
from huggingface_hub import snapshot_download, hf_hub_download, HfApi

def write_json(path, value):
    # 一時ファイルに書いてから置き換える（途中で止まっても壊れたファイルを残さない）
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(value, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

def get_secret(name):
    # ① Colab のシークレット → ② 環境変数 → ③ 手入力（画面に表示しない）の順に探す。トークンは表示も保存もしない
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        value = os.environ.get(name)
    if not value:
        value = getpass.getpass("%s を入力（Hugging Face の Read トークン。入力は画面に表示されない）: " % name)
    value = (value or "").strip()
    if not value:
        raise RuntimeError("%s が空である。Colab の鍵アイコン（シークレット）に追加するか、入力欄に貼ること" % name)
    if not value.startswith("hf_"):
        print("[注意] %s が hf_ で始まっていない。Read トークンか確認すること" % name)
    return value

HF_READ_TOKEN = get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_READ_TOKEN   # モデルの取得にも使う（トークンなしだと遅く、警告も出る）

# トークンが使えるかを先に確認する（ユーザー名だけ表示。トークンは表示しない）
try:
    print("[storage] Hugging Face:", HfApi().whoami(token=HF_READ_TOKEN).get("name"), "としてログインできた")
except Exception as e:
    raise RuntimeError("Hugging Face のトークンが使えない（%s）。Read 権限か、期限切れでないかを確認すること" % type(e).__name__) from e

# 選択した検索のインデックスのフォルダ名
SELECTED_INDEXES = ([BM25_INDEX] if "bm25" in RETRIEVERS else []) + [EMB_INDICES[name] for name in DENSE_NAMES]

t0 = time.time()
try:
    snapshot_download(repo_id=HF_REPO, repo_type="dataset", revision=HF_REVISION, local_dir=LOCAL_DATA,
                      allow_patterns=["data/chunks/chunks-*.jsonl", "data/ATTRIBUTION.jsonl", "data/LICENSE.md", "index/chunk_offsets.npy"] + ["index/%s/*" % folder for folder in SELECTED_INDEXES],
                      token=HF_READ_TOKEN)
except Exception as e:
    if type(e).__name__ in ("GatedRepoError", "RepositoryNotFoundError") or "401" in str(e) or "403" in str(e):
        raise RuntimeError("データセットに入れない（%s）。データセットのページで利用条件に同意したアカウントの Read トークンか確認すること" % type(e).__name__) from e
    raise
print("[storage] 取得した（%.0f s）" % (time.time() - t0))

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("[storage] Colab 以外の環境である。DRIVE_ROOT 以下をそのまま使う")
os.makedirs(WORK_DIR, exist_ok=True)

# 選択したインデックスがそろっていて、作成が完了しているか確認する
INDEX_CONFIGS = {}
for folder in SELECTED_INDEXES:
    cfg_path = os.path.join(INDEX_ROOT, folder, "index_config.json")
    if not os.path.exists(cfg_path):
        raise FileNotFoundError("index/%s/index_config.json がない。BM25_INDEX・EMB_INDICES を確認すること" % folder)
    with open(cfg_path) as f:
        INDEX_CONFIGS[folder] = json.load(f)
    if INDEX_CONFIGS[folder].get("status") != "complete":
        raise RuntimeError("index/%s は作成が完了していない（status=%s）" % (folder, INDEX_CONFIGS[folder].get("status")))
    print("[storage] index/%s: type=%s status=complete" % (folder, INDEX_CONFIGS[folder].get("type")))

[storage] Hugging Face: YAMAHARZ250 としてログインできた


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1236 files:   0%|          | 0/1236 [00:00<?, ?it/s]

[storage] 取得した（2 s）
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[storage] index/bm25-sudachi-c: type=bm25 status=complete
[storage] index/ruri-v3-310m: type=embedding status=complete
[storage] index/bge-m3: type=embedding status=complete


In [ ]:
# # [step1.3 storage]
# # 1) 配布済みデータと選択した検索インデックスをReadトークンで取得する
# # 2) Drive をマウントし、作業フォルダ（実行結果の保存先）を作る
# import json, os, time

# #from huggingface_hub import login
# #login()   # Hugging Face

# #os.environ["HF_TOKEN"] = ""   # 取得時の警告を消す
# from huggingface_hub import snapshot_download, hf_hub_download

# def write_json(path, value):
#     # 一時ファイルに書いてから置き換える（途中で止まっても壊れたファイルを残さない）
#     tmp = path + ".tmp"
#     with open(tmp, "w", encoding="utf-8") as f:
#         json.dump(value, f, ensure_ascii=False, indent=2)
#     os.replace(tmp, path)


# def get_secret(name):
#     try:
#         from google.colab import userdata
#         return userdata.get(name)
#     except ImportError:
#         value = os.environ.get(name)   # Colab 以外の環境
#     except Exception:
#         value = None
#     if value is None:
#         raise RuntimeError("Colab のシークレットに %s がない。左の鍵アイコンから追加し、"
#                            "「ノートブックからのアクセス」を有効にすること" % name)
#     return value


# HF_READ_TOKEN = ""
# os.environ["HF_TOKEN"] = ""   # モデルの取得にも使う（トークンなしだと遅く、警告も出る）
# # 選択した検索のインデックスのフォルダ名
# SELECTED_INDEXES = ([BM25_INDEX] if "bm25" in RETRIEVERS else []) + [EMB_INDICES[name] for name in DENSE_NAMES]

# t0 = time.time()
# snapshot_download(repo_id=HF_REPO, repo_type="dataset", revision=HF_REVISION, local_dir=LOCAL_DATA,
#                   allow_patterns=["data/chunks/chunks-*.jsonl", "data/ATTRIBUTION.jsonl", "data/LICENSE.md", "index/chunk_offsets.npy"] + ["index/%s/*" % folder for folder in SELECTED_INDEXES],
#                   token=HF_READ_TOKEN)
# print("[storage] 取得した（%.0f s）" % (time.time() - t0))


# try:
#     from google.colab import drive
#     drive.mount("/content/drive", force_remount=False)
# except ImportError:
#     print("[storage] Colab 以外の環境である。DRIVE_ROOT 以下をそのまま使う")
# os.makedirs(WORK_DIR, exist_ok=True)

# # 選択したインデックスがそろっていて、作成が完了しているか確認する
# INDEX_CONFIGS = {}
# for folder in SELECTED_INDEXES:
#     cfg_path = os.path.join(INDEX_ROOT, folder, "index_config.json")
#     if not os.path.exists(cfg_path):
#         raise FileNotFoundError("index/%s/index_config.json がない。BM25_INDEX・EMB_INDICES を確認すること" % folder)
#     with open(cfg_path) as f:
#         INDEX_CONFIGS[folder] = json.load(f)
#     if INDEX_CONFIGS[folder].get("status") != "complete":
#         raise RuntimeError("index/%s は作成が完了していない（status=%s）" % (folder, INDEX_CONFIGS[folder].get("status")))
#     print("[storage] index/%s: type=%s status=complete" % (folder, INDEX_CONFIGS[folder].get("type")))


## Step 2: データの読み込み

質問と、チャンクの位置の表（`chunk_offsets.npy`）を読み込む。`QUESTION_LIMIT` を指定した場合は、先頭の質問だけを実行対象（`TARGET_QUESTIONS`）にする。`get_chunks(chunk_ids)` で、指定した chunk_id のチャンクをローカルディスクのファイルから取り出せる。

In [ ]:

# [step2.1 data]  ← 開発用セット専用。本番の質問を Hugging Face から自動取得しないよう、取得処理を外してある
import numpy as np
if not os.path.isfile(QUESTIONS_PATH):
    raise SystemExit("開発用の質問ファイルがない: %s\n開発用の質問（320件・dev0001 形式）を、questions.json の名前でこの場所に置く。"
                     "（このノートブックは、本番の質問を自動取得しない）" % QUESTIONS_PATH)
with open(QUESTIONS_PATH, encoding="utf-8") as f:
    questions = json.load(f)
if not isinstance(questions, list) or len(questions) != 320 or not all(str(q.get("q_id", "")).startswith("dev") for q in questions):
    raise SystemExit("開発用の質問ファイルではない（件数 %s・q_id が dev で始まらない）。本番の質問の可能性があるので止める: %s"
                     % (len(questions) if isinstance(questions, list) else type(questions).__name__, QUESTIONS_PATH))
print("[data] Drive の開発用質問を使う:", QUESTIONS_PATH)
TARGET_QUESTIONS = questions[:QUESTION_LIMIT]   # 実行対象の質問（QUESTION_LIMIT で先頭だけに絞れる）
CHUNK_FILES = sorted(f for f in os.listdir(CHUNKS_DIR) if f.startswith("chunks-") and f.endswith(".jsonl"))
OFFSETS_PATH = os.path.join(INDEX_ROOT, "chunk_offsets.npy")
CHUNK_OFFSETS = np.load(OFFSETS_PATH)
N_CHUNKS = len(CHUNK_OFFSETS)
_handles = {}
def get_chunks(chunk_ids):
    out = []
    for cid in chunk_ids:
        file_no, offset, length = CHUNK_OFFSETS[int(cid)]
        name = CHUNK_FILES[int(file_no)]
        if name not in _handles:
            _handles[name] = open(os.path.join(CHUNKS_DIR, name), "rb")
        fh = _handles[name]
        fh.seek(int(offset))
        out.append(json.loads(fh.read(int(length))))
    return out
print("[data] 質問 %d（実行対象 %d）/ チャンク %d（%d ファイル）"
      % (len(questions), len(TARGET_QUESTIONS), N_CHUNKS, len(CHUNK_FILES)))

[data] Drive の開発用質問を使う: /content/drive/MyDrive/LLM講座最終コンペ_eval/dev_work/questions.json
[data] 質問 320（実行対象 320）/ チャンク 1003139（491 ファイル）


In [ ]:
# # [step2.1 data]
# import numpy as np


# _questions_missing = not os.path.isfile(QUESTIONS_PATH)
# _questions_source = QUESTIONS_PATH
# if _questions_missing:
#     _questions_source = hf_hub_download(
#         repo_id=HF_REPO, repo_type="dataset", revision=HF_REVISION,
#         filename="data/questions.json", token=HF_READ_TOKEN)
# with open(_questions_source, encoding="utf-8") as f:
#     questions = json.load(f)
# if _questions_missing:
#     write_json(QUESTIONS_PATH, questions)
#     print("[data] Hugging Face から質問を取得して保存した:", QUESTIONS_PATH)
# else:
#     print("[data] Drive の質問ファイルを使う:", QUESTIONS_PATH)

# TARGET_QUESTIONS = questions[:QUESTION_LIMIT]   # 実行対象の質問（QUESTION_LIMIT で先頭だけに絞れる）

# CHUNK_FILES = sorted(f for f in os.listdir(CHUNKS_DIR) if f.startswith("chunks-") and f.endswith(".jsonl"))


# OFFSETS_PATH = os.path.join(INDEX_ROOT, "chunk_offsets.npy")
# CHUNK_OFFSETS = np.load(OFFSETS_PATH)
# N_CHUNKS = len(CHUNK_OFFSETS)
# _handles = {}


# def get_chunks(chunk_ids):
#     out = []
#     for cid in chunk_ids:
#         file_no, offset, length = CHUNK_OFFSETS[int(cid)]
#         name = CHUNK_FILES[int(file_no)]
#         if name not in _handles:
#             _handles[name] = open(os.path.join(CHUNKS_DIR, name), "rb")
#         fh = _handles[name]
#         fh.seek(int(offset))
#         out.append(json.loads(fh.read(int(length))))
#     return out


# print("[data] 質問 %d（実行対象 %d）/ チャンク %d（%d ファイル）"
#       % (len(questions), len(TARGET_QUESTIONS), N_CHUNKS, len(CHUNK_FILES)))


## Step 3: 配布済みインデックスの読み込み

### 3.1 BM25

配布済みの転置インデックスを読み込む。`BM25_K1`・`BM25_B` は、読み込み時に各語の重みへ反映する。`RETRIEVERS` に `bm25` がなければ読み込まない。

In [ ]:
# [step3.1 bm25]

_SUDACHI = None


def sudachi_tokenize(text):
    # SplitMode C で分割して辞書形にする。空白と補助記号（句読点など）は除く
    global _SUDACHI
    from sudachipy import Dictionary, Tokenizer
    if _SUDACHI is None:
        _SUDACHI = Dictionary().create()
    return [m.dictionary_form() for m in _SUDACHI.tokenize(text, Tokenizer.SplitMode.C)
            if m.part_of_speech()[0] not in {"空白", "補助記号"}]


class BM25:
    def __init__(self, path, k1=BM25_K1, b=BM25_B):
        z = np.load(os.path.join(path, "bm25_tf.npz"))
        self.indptr, self.indices = z["indptr"], z["indices"]
        tf = z["data"].astype(np.float32)
        doclen = np.load(os.path.join(path, "bm25_doclen.npy")).astype(np.float32)
        with open(os.path.join(path, "bm25_vocab.txt")) as f:
            self.vocab = {t: i for i, t in enumerate(f.read().split("\n")[:-1])}
        n = len(doclen)
        df = np.diff(self.indptr)
        idf = np.maximum(np.log((n - df + 0.5) / (df + 0.5)), 1e-6).astype(np.float32)
        norm = k1 * (1 - b + b * doclen / doclen.mean())
        term = np.repeat(np.arange(len(df), dtype=np.int32), df)   # 各要素がどの語か
        self.weights = idf[term] * tf * (k1 + 1) / (tf + norm[self.indices])
        self.n = n

    def search(self, query, k=BM25_TOP_K):
        # 戻り値: [(chunk_id, スコア), ...] をスコアの高い順に
        terms = {self.vocab[t] for t in sudachi_tokenize(query) if t in self.vocab}
        if not terms:
            return []
        scores = np.zeros(self.n, dtype=np.float32)
        for j in terms:
            s, e = self.indptr[j], self.indptr[j + 1]
            scores[self.indices[s:e]] += self.weights[s:e]
        k = min(k, int((scores > 0).sum()))
        if k == 0:
            return []
        top = np.argpartition(-scores, k - 1)[:k]
        return [(int(d), float(scores[d])) for d in sorted(top.tolist(), key=lambda d: (-scores[d], d))]


bm25 = None
if "bm25" in RETRIEVERS:
    t0 = time.time()
    bm25 = BM25(os.path.join(INDEX_ROOT, BM25_INDEX))
    print("[bm25] %s: %dチャンク / %d語（%.0f s）" % (BM25_INDEX, bm25.n, len(bm25.vocab), time.time() - t0))
else:
    print("[bm25] 使わない（RETRIEVERS に bm25 がない）")

[bm25] bm25-sudachi-c: 1003139チャンク / 1153164語（5 s）


### 3.2 選択した埋め込みモデルの設定を照合する

Ruri は平均、BGE-M3 は CLS、Qwen は最後の有効トークンでプーリングする。
質問の埋め込みには設定セルの `EMBED_CONFIGS` を使い、各インデックスの設定 JSON（`index_config.json`）と項目ごとに照合する。
文書ベクトル作成時と違う項目は `[警告]`、質問側だけの項目（`query_prefix`・`max_length`）の変更は `[変更]` として表示する。

In [ ]:
# [step3.2 embedding]
import gc


def pick_device():
    if DEVICE != "auto":
        return DEVICE
    return "cuda" if torch.cuda.is_available() else "cpu"


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


class Encoder:
    def __init__(self, spec):
        from transformers import AutoModel, AutoTokenizer
        self.spec, self.prefix, self.device = spec, spec["query_prefix"], pick_device()
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        kw = dict(revision=spec["revision"], trust_remote_code=spec["trust_remote_code"])
        self.tokenizer = AutoTokenizer.from_pretrained(spec["model"], **kw)
        self.tokenizer.padding_side = "right"  # CLS/mean/lastすべてで文書作成時の方式と揃える
        if self.tokenizer.pad_token_id is None:
            if self.tokenizer.eos_token_id is None:
                raise ValueError("padding 用のトークンがない")
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModel.from_pretrained(spec["model"], dtype=dtype, attn_implementation="sdpa", **kw).to(self.device).eval()

    @torch.inference_mode()
    def encode(self, texts, batch_size=EMBED_BATCH_SIZE):
        out = []
        for i in range(0, len(texts), batch_size):
            batch = [self.prefix + t for t in texts[i:i + batch_size]]
            tok = self.tokenizer(batch, padding=True, truncation=True,
                                 max_length=self.spec["max_length"], return_tensors="pt").to(self.device)
            hidden = self.model(**tok).last_hidden_state
            mask = tok["attention_mask"]
            if self.spec["pooling"] == "mean":
                m = mask.unsqueeze(-1).to(hidden.dtype)
                vec = (hidden * m).sum(1) / m.sum(1).clamp_min(1)
            elif self.spec["pooling"] == "cls":
                vec = hidden[:, 0]
            else:   # "last"（Qwen）: 右詰めのpaddingなので、最後の有効トークンは有効トークン数 - 1 の位置
                last = mask.sum(1) - 1
                if (last < 0).any():
                    raise ValueError("有効トークンがない入力がある")
                vec = hidden[torch.arange(len(batch), device=hidden.device), last]
            vec = vec.float()
            if self.spec["normalize"]:
                vec = torch.nn.functional.normalize(vec, p=2, dim=1)
            out.append(vec.cpu().numpy())
        return np.concatenate(out)


QUERY_ONLY_KEYS = {"query_prefix", "max_length"}   # 文書ベクトルと揃えなくてよい項目

# 選択した全モデルについて、EMBED_CONFIGS とインデックス作成時の設定を照合する。実際のモデル読み込みは次の検索セル。
DENSE_INDEXES = {}
for name in DENSE_NAMES:
    folder = os.path.join(INDEX_ROOT, EMB_INDICES[name])
    spec, built = EMBED_CONFIGS[name], INDEX_CONFIGS[EMB_INDICES[name]]["embed"]   # [step1.3 storage] で読み込み済み
    if spec["pooling"] not in ("mean", "cls", "last"):
        raise ValueError('EMBED_CONFIGS["%s"]["pooling"] は "mean" | "cls" | "last" のいずれかにすること: %r'
                         % (name, spec["pooling"]))
    DENSE_INDEXES[name] = {"folder": folder, "embed": spec, "index_embed": built}
    print("[embedding]", name, folder, spec["model"], spec["pooling"])
    for key in spec:
        if spec[key] == built.get(key):
            continue
        if key in QUERY_ONLY_KEYS:
            print("  [変更] %s: 質問側の %s を変えている（インデックス %r / この実行 %r）" % (name, key, built.get(key), spec[key]))
        else:
            print("  [警告] %s: %s が文書ベクトル作成時と違う（インデックス %r / この実行 %r）。検索精度が大きく下がる"
                  % (name, key, built.get(key), spec[key]))


[embedding] ruri /content/rag_data/index/ruri-v3-310m cl-nagoya/ruri-v3-310m mean
[embedding] bge /content/rag_data/index/bge-m3 BAAI/bge-m3 cls


## Step 4: 埋め込み検索とリランカーの準備

実行対象の質問を各モデルで埋め込み、文書ベクトルをシャード単位で検索して上位候補を保持する。
モデル間でベクトルは混ぜず、各モデルの検索順位は Step 5 で定義する `fuse` で統合する。
リランカーを使う設定なら、続くセルでリランカーを読み込む。

In [ ]:
# [step4.1 dense-search]
import re


def document_shards(folder):
    # 数字部分で並べるだけ。配列は検索時に各ファイルを1回ずつ読み込む。
    found = []
    for name in os.listdir(folder):
        match = re.fullmatch(r"emb_(\d+)\.npy", name)
        if match:
            found.append((int(match.group(1)), name))
    return [name for _, name in sorted(found)]


def search_dense_index(name, queries):
    # 検索モデルは1つずつ、文書ベクトルは1シャードずつ読み込む。
    if not queries:
        return {}
    entry = DENSE_INDEXES[name]
    folder, spec = entry["folder"], entry["embed"]
    files = document_shards(folder)
    device = pick_device()
    k = min(RETRIEVER_TOP_K[name], N_CHUNKS)
    encoder = q_gpu = block = sim = best_scores = best_ids = None
    try:
        encoder = Encoder(spec)
        q_array = encoder.encode(queries)
        # 質問ベクトルを作った後はモデルを解放し、文書検索のメモリを空ける。
        del encoder
        free_gpu()
        q_gpu = torch.from_numpy(q_array).to(device)
        best_scores = torch.full((len(queries), k), -torch.inf, device=device)
        best_ids = torch.full((len(queries), k), -1, device=device, dtype=torch.long)
        offset = 0
        for file in files:
            arr = np.load(os.path.join(folder, file), allow_pickle=False)
            block = torch.from_numpy(arr).to(device, dtype=torch.float32)
            for start in range(0, len(queries), DENSE_QUERY_BATCH_SIZE):
                stop = min(start + DENSE_QUERY_BATCH_SIZE, len(queries))
                sim = q_gpu[start:stop] @ block.T
                vals, ids = torch.topk(sim, min(k, len(arr)), dim=1)
                ids += offset
                candidates = torch.cat((best_scores[start:stop], vals), dim=1)
                candidate_ids = torch.cat((best_ids[start:stop], ids), dim=1)
                selected, order = torch.topk(candidates, k, dim=1)
                best_scores[start:stop] = selected
                best_ids[start:stop] = candidate_ids.gather(1, order)
                del sim, vals, ids, candidates, candidate_ids, selected, order
            offset += len(arr)
            del arr, block
        ids_list = best_ids.cpu().tolist()
        scores_list = best_scores.cpu().tolist()
        return {query: list(zip(ids_list[i], scores_list[i])) for i, query in enumerate(queries)}
    finally:
        encoder = q_gpu = block = sim = best_scores = best_ids = None
        free_gpu()


# 実行対象の質問だけ検索し、上位k件の結果を保持する。
# 同じ質問文は一度だけ処理する。
_query_texts = list(dict.fromkeys(q["question"] for q in TARGET_QUESTIONS))
DENSE_RESULTS = {}
for name in DENSE_NAMES:
    t0 = time.time()
    DENSE_RESULTS[name] = search_dense_index(name, _query_texts)
    print("[dense]", name, len(_query_texts), "問 / top %d / %.0f s" % (RETRIEVER_TOP_K[name], time.time() - t0))


Loading weights:   0%|          | 0/152 [00:00<?, ?it/s]

[dense] ruri 320 問 / top 100 / 42 s


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[dense] bge 320 問 / top 100 / 24 s


In [ ]:
# [step4.2 reranker]
class Reranker:
    # Qwen3-Reranker の公式の書式。質問と文書を渡し、次の語が "yes" になる確率をスコアにする
    QWEN3_PREFIX = ("<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and "
                    'the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')
    QWEN3_SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

    def __init__(self, spec):
        from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer
        self.spec, self.device = spec, pick_device()
        self.fp32 = self.device == "cpu"
        dtype = torch.float32 if self.fp32 else torch.float16
        if spec["kind"] == "qwen3":
            self.tokenizer = AutoTokenizer.from_pretrained(spec["model"], revision=spec["revision"], padding_side="left")
            self.model = AutoModelForCausalLM.from_pretrained(
                spec["model"], revision=spec["revision"], dtype=dtype).to(self.device).eval()
            self.no_yes = [self.tokenizer.convert_tokens_to_ids(t) for t in ("no", "yes")]
            self.prefix = self.tokenizer.encode(self.QWEN3_PREFIX, add_special_tokens=False)
            self.suffix = self.tokenizer.encode(self.QWEN3_SUFFIX, add_special_tokens=False)
        elif spec["kind"] == "cross_encoder":
            self.tokenizer = AutoTokenizer.from_pretrained(spec["model"], revision=spec["revision"])
            self.model = AutoModelForSequenceClassification.from_pretrained(
                spec["model"], revision=spec["revision"], dtype=dtype).to(self.device).eval()
        else:
            raise ValueError("RERANKER の kind が不明である: %r" % spec["kind"])

    def _tokenize(self, query, batch):
        if self.spec["kind"] == "qwen3":
            texts = ["<Instruct>: %s\n<Query>: %s\n<Document>: %s" % (self.spec["instruction"], query, p) for p in batch]
            enc = self.tokenizer(texts, padding=False, truncation="longest_first", return_attention_mask=False,
                                 max_length=self.spec["max_length"] - len(self.prefix) - len(self.suffix))
            enc["input_ids"] = [self.prefix + ids + self.suffix for ids in enc["input_ids"]]
            return self.tokenizer.pad(enc, padding=True, return_tensors="pt").to(self.device)
        return self.tokenizer([query] * len(batch), batch, padding=True, truncation=True,
                              max_length=self.spec["max_length"], return_tensors="pt").to(self.device)

    def _forward(self, tok):
        if self.spec["kind"] == "qwen3":
            # 最後の位置の "no" と "yes" の logits だけを使う（全位置の語彙 logits は計算しない）
            logits = self.model(**tok, logits_to_keep=1).logits[:, -1, self.no_yes].float()
            return logits, torch.softmax(logits, dim=1)[:, 1]
        logits = self.model(**tok).logits.float()
        return logits, logits[:, 0] if logits.shape[-1] == 1 else logits[:, -1]

    @torch.inference_mode()
    def score(self, query, passages):
        scores = []
        for i in range(0, len(passages), self.spec["batch_size"]):
            tok = self._tokenize(query, passages[i:i + self.spec["batch_size"]])
            logits, col = self._forward(tok)
            if not torch.isfinite(logits).all() and not self.fp32:
                # float16 で NaN / inf が出たときだけ float32 で計算し直す
                self.model.float()
                self.fp32 = True
                logits, col = self._forward(tok)
            scores.extend(col.cpu().tolist())
        return scores


reranker = Reranker(RERANKER) if USE_RERANKER else None
print("[reranker]", "%s（%s）" % (RERANKER_NAME, RERANKER["model"]) if USE_RERANKER else "使わない")

Loading weights:   0%|          | 0/156 [00:00<?, ?it/s]

[reranker] ruri（cl-nagoya/ruri-v3-reranker-310m）


## Step 5: 検索・統合・リランクの定義

**検索**（`retrieve`）: 1問ごとに次の順で処理し、生成モデルに渡す資料の dict のリストを返す。

1. **各検索**: `RETRIEVERS` に入っている検索を実行する。どの検索も `[(chunk_id, スコア), ...]` をスコアの高い順に返す
   - `bm25`（`bm25.search`）: 上位 `BM25_TOP_K` 件
   - `ruri` / `bge` / `qwen`: Step 4 で事前に検索した上位 `RURI_TOP_K` / `BGE_TOP_K` / `QWEN_TOP_K` 件を使う
2. **統合**（`fuse`）: 検索が1つだけならその結果をそのまま使う。2つ以上なら `FUSION` の方法でまとめ、上位 `FUSED_TOP_K` 件を残す。ベースラインは RRF（`score = Σ 1 / (RRF_K + 順位)`）
3. **リランク**: リランカーを使う場合は、統合結果の上位 `RERANK_CANDIDATES` 件について質問と `タイトル\n本文` の関連度を計算して並べ替え、上位 `CONTEXT_TOP_K` 件を残す。使わない場合は、統合結果の上位 `CONTEXT_TOP_K` 件をそのまま使う

検索を足すときは、同じ形の戻り値を返す関数を書いて `SEARCHERS` に登録する。統合方法を足すときは、`FUSIONS` に関数を登録して `FUSION` で名前を指定する。各検索のスコアも渡しているので、順位ではなくスコアを使う方法も書ける。

`retrieve()` の戻り値は、最終順位順の `[{"chunk_id": ..., "title": ..., "text": ...}, ...]` である。リランカーの有無にかかわらず同じ形になる。

In [ ]:
# [step5.1 pipeline]

SEARCHERS = {}
if "bm25" in RETRIEVERS:
    SEARCHERS["bm25"] = lambda query: bm25.search(query)
for name in DENSE_NAMES:
    SEARCHERS[name] = DENSE_RESULTS[name].__getitem__


def rrf(results, k=RRF_K):
    # results: {検索名: [(chunk_id, スコア), ...]}。順位だけを使う
    scores = {}
    for ranked in results.values():
        for rank, (cid, _) in enumerate(ranked, 1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=lambda cid: (-scores[cid], cid))


# 統合方法の一覧。FUSION で名前を指定する。戻り値は chunk_id のリスト（関連が高い順）
FUSIONS = {
    "rrf": rrf,
}


def fuse(results):
    if len(results) == 1:   # 検索が1つなら統合しない
        ranked = [cid for cid, _ in next(iter(results.values()))]
    else:
        ranked = FUSIONS[FUSION](results)
    return ranked[:FUSED_TOP_K]


def retrieve(query):
    # 検索側の出口は、最終順位順のchunk_id・title・textを持つdictのリスト。
    fused = fuse({name: SEARCHERS[name](query) for name in RETRIEVERS})
    candidate_ids = fused[:RERANK_CANDIDATES] if reranker is not None else fused[:CONTEXT_TOP_K]
    contexts = get_chunks(candidate_ids)
    if reranker is not None and contexts:
        scores = reranker.score(query, [c["title"] + "\n" + c["text"] for c in contexts])
        order = sorted(range(len(contexts)), key=lambda i: (-scores[i], contexts[i]["chunk_id"]))
        contexts = [contexts[i] for i in order[:CONTEXT_TOP_K]]
    return [{"chunk_id": c["chunk_id"], "title": c["title"], "text": c["text"]} for c in contexts]


## Step 6: 検索結果を共通形式にまとめる

検索設定を `retrieval_config.json` に保存してから、実行対象の質問（`TARGET_QUESTIONS`）について `retrieve()` を実行する。`retrieve()` が検索・統合・リランク・本文取得まで行い、最終順位順の資料を返すので、ここでは質問 ID と質問文を付けて次の `generation_inputs`（dict のリスト）にまとめる。`generation_inputs` 自体はファイルに保存しない。

```python
generation_inputs = [
    {
        "q_id": "q001",
        "question": "質問文",
        "contexts": [
            {"chunk_id": 456, "title": "記事タイトル", "text": "本文"}
        ],
    }
]
```

`contexts` は、回答生成モデルへ提示する順に並ぶ。リランカーの有無にかかわらず同じ形式である。
独自の検索処理を使う場合も、この形式のデータを用意すれば Step 7 以降を実行できる。形式は続くセル（`[step6.2 validate]`）で検証する。
質問数・資料数はこのインターフェースでは固定しない。資料が0件の場合も扱える。

In [ ]:
# [step6.1 retrieval]

retrieval_config = {
    "bm25_index": BM25_INDEX if "bm25" in RETRIEVERS else None,
    "dense_indexes": {name: {"index": EMB_INDICES[name], "embed": DENSE_INDEXES[name]["embed"],
                             "index_embed": DENSE_INDEXES[name]["index_embed"]} for name in DENSE_NAMES},
    "reranker": RERANKER if USE_RERANKER else None,
    "params": {"retrievers": RETRIEVERS, "fusion": FUSION if len(RETRIEVERS) > 1 else None, "bm25_k1": BM25_K1, "bm25_b": BM25_B,
               "retriever_top_k": {name: RETRIEVER_TOP_K[name] for name in RETRIEVERS}, "rrf_k": RRF_K, "fused_top_k": FUSED_TOP_K,
               "rerank_candidates": RERANK_CANDIDATES, "context_top_k": CONTEXT_TOP_K},
}
os.makedirs(RUN_DIR, exist_ok=True)
write_json(os.path.join(RUN_DIR, "retrieval_config.json"), dict(retrieval_config, env=ENV))
# 検索側の最終出力。このリスト以降は検索方式・リランカーの有無に依存しない。
generation_inputs = [
    {"q_id": q["q_id"], "question": q["question"], "contexts": retrieve(q["question"])}
    for q in TARGET_QUESTIONS
]
print("検索完了:", len(generation_inputs))


検索完了: 320


In [ ]:
# [step6.2 validate]

# 検索側と回答生成側の共通インターフェース。件数や検索方法は固定しない。
def validate_generation_inputs(items):
    if not isinstance(items, list):
        raise ValueError("generation_inputs は dict のリストにすること")
    seen_questions = set()
    for i, item in enumerate(items):
        label = "generation_inputs[%d]" % i
        if not isinstance(item, dict) or not {"q_id", "question", "contexts"} <= item.keys():
            raise ValueError(label + " に q_id・question・contexts が必要である")
        qid = item["q_id"]
        if not isinstance(qid, str) or not qid or qid in seen_questions:
            raise ValueError(label + ".q_id は重複のない空でない文字列にすること")
        seen_questions.add(qid)
        if not isinstance(item["question"], str):
            raise ValueError(label + ".question は文字列にすること")
        if not isinstance(item["contexts"], list):
            raise ValueError(label + ".contexts は dict のリストにすること")
        seen_chunks = set()
        for j, context in enumerate(item["contexts"]):
            path = label + ".contexts[%d]" % j
            if not isinstance(context, dict) or not {"chunk_id", "title", "text"} <= context.keys():
                raise ValueError(path + " に chunk_id・title・text が必要である")
            cid = context["chunk_id"]
            if type(cid) is not int or cid < 0 or cid in seen_chunks:
                raise ValueError(path + ".chunk_id は質問内で重複のない非負整数にすること")
            seen_chunks.add(cid)
            if not isinstance(context["title"], str) or not isinstance(context["text"], str):
                raise ValueError(path + " の title・text は文字列にすること")
    return items


validate_generation_inputs(generation_inputs)

[{'q_id': 'dev0001',
  'question': '「鱸」の読み方は？',
  'contexts': [{'chunk_id': 253595,
    'title': 'スズキ (魚)',
    'text': 'スズキ（鱸、松江魚、紫鰓魚、学名：Lateolabrax japonicus） は、ホタルジャコ目スズキ科に属する魚。海岸近くや河川に生息する大型の肉食魚で、食用や釣りの対象魚として人気がある。日本では成長につれて呼び名が変わる出世魚である。秋の季語。\n名称\n由来\n「スズキ」の名の由来については諸説ある。\nススミ説（出世魚で出世に進むことに由来するとの説）\nスサマジグチ説（口が凄まじく大きいことに由来するとの説）\nススケ説（鱗がすすけた色であることに由来するとの説）\nススギ説（鱗がすすいだように白いことに由来するとの説）\nススジ説（すずしく清らかな身であることに由来するとの説）\n出世魚\n日本においてスズキはいわゆる出世魚で、成長とともに呼び名が変わる。ただし、地方によって呼び名は様々に異なり、統一的な定義はない。'},
   {'chunk_id': 912218,
    'title': '西郷どん (NHK大河ドラマ)',
    'text': '。\n鱸成信（すずき なりのぶ）\n演：小久保寿人\n警視抜刀隊隊員。\n木戸松子（きど まつこ）\n演：白須慶子\n木戸孝允の妻。\n諸外国\nハリス\n演：ブレイク・クロフォード\nアメリカ総領事。名はタウンゼント。\nヒュースケン\n演：セルゲイ・イワノフ\nハリスの通弁官。\nリチャードソン\n演：ノーアム・カッツ\n在日イギリス人。\nロッシュ\n演：ジル・ボーフィ\nフランス公使。\nカシヨン\n演：ロハン・エスピネ\nロッシュの通弁官。\nパークス\n演：セイン・カミュ\nイギリス公使。名はハリー。\nアーネスト・サトウ\n演：スティーブ・ワイリー\nパークスの通弁官。\nパークス夫人\n演：ハンナ・グレース\nパークスの妻。\nウィリス\n演：ネイサン・ベリー\nイギリス人医師。\nスタッフ\n原作：林真理子『西郷どん!』\n脚本：中園ミホ\n音楽：富貴晴美\n語り：西田敏行\n副音声解説：宗方脩'},
   {'chunk_id': 912

In [ ]:
# 【開発用検索結果の保存セル】R01 のコピーの [step6.2 validate] の直後に1回だけ置く（生成が走らないよう、最後に止める）
import json, os, time, collections
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
OUT = EVAL_ROOT + "/dev_contexts.jsonl"

# --- 取り違え防止（本番の質問で保存しない）---
assert "LLM講座最終コンペ_eval" in str(WORK_DIR), "WORK_DIR が評価用フォルダではない: " + str(WORK_DIR)
assert "LLM講座最終コンペ_work" not in str(WORK_DIR), "WORK_DIR が本番用フォルダ: " + str(WORK_DIR)
assert CONTEXT_TOP_K == 10, "CONTEXT_TOP_K は 10 にする（評価ノートブックが 5・7・10 件に切り出す）: %s" % CONTEXT_TOP_K

gi = generation_inputs
rows = list(gi.values()) if isinstance(gi, dict) else list(gi)
r0 = rows[0]
print("件数:", len(rows), "/ 1件目のキー:", sorted(r0.keys()) if isinstance(r0, dict) else type(r0))

def pick(r, *names):
    for n in names:
        if n in r:
            return r[n]
    raise SystemExit("キー %s が見つからない。上の「1件目のキー」の行だけを貼る（中身は貼らない）" % (names,))

out = []
for r in rows:
    ctx = pick(r, "contexts", "chunks", "retrieved_chunks")
    out.append({"q_id": pick(r, "q_id", "id"), "question": pick(r, "question", "query"),
                "contexts": [{"chunk_id": c["chunk_id"], "title": c["title"], "text": c["text"]} for c in ctx]})

# --- 保存（すでにあるファイルは別名に退避。上書きしない）---
if os.path.exists(OUT):
    bak = OUT + ".bak_" + time.strftime("%Y%m%d_%H%M%S")
    os.rename(OUT, bak)
    print("すでにあった dev_contexts.jsonl を退避:", bak)
with open(OUT, "w", encoding="utf-8") as f:
    for o in out:
        f.write(json.dumps(o, ensure_ascii=False) + "\n")
print("保存:", OUT, "/", len(out), "問 / 1問あたりの資料:", dict(collections.Counter(len(o["contexts"]) for o in out)))
raise SystemExit("開発用セットの検索結果を保存した。ここで止める（生成は動かさない）")

件数: 320 / 1件目のキー: ['contexts', 'q_id', 'question']
保存: /content/drive/MyDrive/LLM講座最終コンペ_eval/dev_contexts.jsonl / 320 問 / 1問あたりの資料: {10: 320}


SystemExit: 開発用セットの検索結果を保存した。ここで止める（生成は動かさない）

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Step 7: 推論設定と回答生成モデルの起動

次のセルで、回答生成のプロンプトとハイパーパラメータをまとめて指定する。`answer_prompt()` の文章や資料の並べ方、`GENERATOR` の生成設定、`SEED`、`GEN_BATCH_SIZE` を変更できる。モデル ID と revision は利用ルールにより固定である。

検索方法を変える場合は検索側を編集し、最終出力を `generation_inputs` の共通形式に合わせる。

In [ ]:
# [step7.1 generation-config] 編集可: プロンプト・生成ハイパーパラメータ
# 回答生成モデルはコンペのルールで Qwen/Qwen3.5-0.8B に固定（model と revision は変更不可）
'''
GENERATOR = {
    "model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "max_new_tokens": 1024,
    # サンプリング: モデルカードの推奨値（non-thinking モード・テキスト）
    "temperature": 1.0,
    "top_p": 1.0,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 2.0,
    "repetition_penalty": 1.0,
    "max_model_len": 8192,        # 入力+出力の最大トークン数
    "gpu_memory_utilization": 0.5,
    "dtype": "float16",           # T4 は bfloat16 非対応
    "enforce_eager": True,        # True: CUDA グラフを使わない（Qwen3.5 は T4 で False にすると初期化に失敗する）
    "enable_thinking": False,     # True: thinkingを有効にする
    "use_structured_outputs": True, # False: 出力JSONスキーマの制約を使わない
    "text_only": True,            # True: 画像・動画の入力を無効にする（画像も扱えるモデルの起動が速くなる）
}
'''

GENERATOR = {
    "model": "YAMAHARZ250/qwen3.5-0.8b-rag-lora-final",
    "revision": "9a4d0723a493aa3c2b9570b065744ea16f4a9237",
    "max_new_tokens": 64,
    # サンプリング: 温度0（毎回いちばん確率の高い語を選ぶ）・繰り返しペナルティなし
    "temperature": 0.0,
    "top_p": 1.0,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 0.0,
    "repetition_penalty": 1.0,
    "max_model_len": 8192,        # 入力+出力の最大トークン数
    "gpu_memory_utilization": 0.5,
    "dtype": "float16",           # T4 は bfloat16 非対応
    "enforce_eager": True,        # True: CUDA グラフを使わない（Qwen3.5 は T4 で False にすると初期化に失敗する）
    "enable_thinking": False,     # True: thinkingを有効にする
    "use_structured_outputs": True, # False: 出力JSONスキーマの制約を使わない
    "text_only": True,            # True: 画像・動画の入力を無効にする（画像も扱えるモデルの起動が速くなる）
}



SEED = 42
GEN_BATCH_SIZE = 10               # 何問ずつまとめて生成するか（1バッチごとに提出ファイルへ途中結果を保存）



'''
def answer_prompt(question, contexts):
    context_text = "\n\n".join("[chunk_id=%d] %s\n%s" % (c["chunk_id"], c["title"], c["text"]) for c in contexts)
    return (
        "次の資料だけを根拠に質問に答えてください。資料から答えを確定できない場合は"
        "answer を「資料からは特定できません」、chunk_id を null としてください。\n"
        "回答の説明は含めず、JSONだけを返してください。キーは answer と chunk_id の2つです。"
        "answer は短い回答文字列、chunk_id は回答の根拠として最も直接的な資料の chunk_id(整数)を1つだけにしてください。\n"
        '形式: {"answer":"回答文字列","chunk_id":整数}\n\n'
        "資料:\n" + context_text + "\n\n質問: " + question
    )

'''


def answer_prompt(question, contexts):
    context_text = "\n\n".join("[chunk_id=%d] %s\n%s" % (c["chunk_id"], c["title"], c["text"]) for c in contexts)
    return (
        "質問: " + question + "\n\n"
        "次の資料を根拠に、上の質問に短く答えてください。\n"
        "答え方のきまり:\n"
        "- answer には、質問の答えになる語句だけを書く（人名・地名・作品名・数値・日付など）。\n"
        "- 文にしない。「〜です」「〜である」を付けない。説明・補足・括弧書き・読み仮名・英語表記を付けない。\n"
        "- 「どちら」「どっち」と聞かれたら、選んだほうの名前だけを書く。\n"
        "- 「はい」か「いいえ」で答える質問には、「はい」か「いいえ」だけを書く。\n"
        "- 資料のどこにも答えが書かれていない場合だけ、answer を「資料からは特定できません」、chunk_id を null にする。\n"
        "chunk_id には、答えが書かれている資料の chunk_id(整数)を1つだけ書く。\n"
        '形式: {"answer":"答えの語句","chunk_id":整数}\n\n'
        "資料:\n" + context_text + "\n\n質問（再掲）: " + question
    )


print("GENERATOR:", GENERATOR["model"], "/ batch:", GEN_BATCH_SIZE)



In [ ]:
# 【提出前の照合セル】[step7.1 generation-config] の直後に1回実行する
# ① 提出用のプロンプトが、学習時と同じか（md5）② GENERATOR の model・revision が、本番モデルの値か
# 一致しなければ、ここで止まる（「すべてを実行」でも、生成に進まない）
import hashlib
TRAIN_MD5 = "1e990717ef9ebb51535b233deec1503e"                      # 学習セルが表示した値（2026-11-08 の本番学習）
WANT_MODEL = "YAMAHARZ250/qwen3.5-0.8b-rag-lora-final"
WANT_REV = "9a4d0723a493aa3c2b9570b065744ea16f4a9237"
now = hashlib.md5(answer_prompt("Q", [{"chunk_id": 1, "title": "T", "text": "X"}]).encode("utf-8")).hexdigest()
print("提出用の md5:", now, "/ 学習時の md5:", TRAIN_MD5, "→", "一致" if now == TRAIN_MD5 else "【不一致】")
bad = []
if now != TRAIN_MD5:
    bad.append("プロンプトの md5 が学習時と違う")
g = globals().get("GENERATOR")
if isinstance(g, dict) and "model" in g:
    print("GENERATOR: model =", g.get("model"), "/ revision =", g.get("revision"))
    if g.get("model") != WANT_MODEL or g.get("revision") != WANT_REV:
        bad.append("GENERATOR の model・revision が本番モデルの値と違う")
else:
    print("[注意] GENERATOR が辞書として見つからない。[step7.1] の有効な方（三連引用符の外）の model・revision を、目で確認する")
if bad:
    raise SystemExit("提出しない: " + " / ".join(bad) + "。[step7.1] を直して、そこから実行し直す")
print("OK: プロンプトも GENERATOR も、本番モデルと学習時に一致している")

## ここから下は編集不要

ここから下のセルは、そのまま実行する。

### モデル起動

生成設定を `generation_config.json` に保存し、リランカーと BM25 を解放してから回答生成モデルを起動する。プロンプトとハイパーパラメータは、上の推論設定セルの値を使う。

In [ ]:
# [step7.2 generator] 編集不要

# [step7.1 generation-config] の生成設定を、検索設定とは別に保存する。
write_json(os.path.join(RUN_DIR, "generation_config.json"), {
    "generator": GENERATOR, "seed": SEED, "batch_size": GEN_BATCH_SIZE,
})


def answer_schema(contexts):
    # 出力を {"answer": 文字列, "chunk_id": 渡した資料の chunk_id か null} の JSON に制約する
    return {
        "type": "object",
        "properties": {
            "answer": {"type": "string"},
            "chunk_id": {"enum": [c["chunk_id"] for c in contexts] + [None]},
        },
        "required": ["answer", "chunk_id"],
        "additionalProperties": False,
    }


def parse_answer(raw):
    # 生成モデルの出力から (answer, chunk_id) を取り出す。読み取れないものは None
    try:
        payload = json.loads(raw)
    except json.JSONDecodeError:
        payload = None
    if isinstance(payload, dict):
        answer, cid = payload.get("answer"), payload.get("chunk_id")
    else:
        # max_new_tokens に達して JSON が途中で切れた場合: answer を書けたところまで取り出す
        m = re.match(r'\s*\{\s*"answer"\s*:\s*"((?:[^"\\]|\\.)*)', raw)
        answer, cid = None, None
        if m:
            try:
                answer = json.loads('"' + m.group(1) + '"')
            except json.JSONDecodeError:
                answer = m.group(1)
    if not isinstance(answer, str):
        answer = None
    if isinstance(cid, str) and cid.strip().isdigit():   # "123" のような数字だけの文字列は整数に直す
        cid = int(cid.strip())
    if not isinstance(cid, int) or isinstance(cid, bool) or cid < 0:
        cid = None
    return answer, cid


class VLLMGenerator:
    def __init__(self, spec):
        self.spec = spec
        import contextlib, importlib
        # Colab 向けの回避策: Colab の出力は fileno() に対応しないため、vLLM の標準出力の抑制を無効にする
        try:
            m = importlib.import_module("vllm.utils.system_utils")
            if hasattr(m, "suppress_stdout"):
                m.suppress_stdout = contextlib.nullcontext
        except ImportError:
            pass
        from vllm import LLM, SamplingParams
        from vllm.sampling_params import StructuredOutputsParams
        extra = {"limit_mm_per_prompt": {"image": 0, "video": 0}} if spec["text_only"] else {}
        self.llm = LLM(model=spec["model"], revision=spec["revision"], dtype=spec["dtype"],
                       max_model_len=spec["max_model_len"], gpu_memory_utilization=spec["gpu_memory_utilization"],
                       enforce_eager=spec["enforce_eager"], seed=SEED, **extra)
        self.sampling = dict(max_tokens=spec["max_new_tokens"], seed=SEED, **{k: spec[k] for k in (
            "temperature", "top_p", "top_k", "min_p", "presence_penalty", "repetition_penalty")})
        self.SamplingParams, self.StructuredOutputsParams = SamplingParams, StructuredOutputsParams

    def generate(self, prompts, schemas):
        # schemas: プロンプトごとの JSON スキーマ
        params = [self.SamplingParams(
            **self.sampling, structured_outputs=self.StructuredOutputsParams(json=s) if self.spec["use_structured_outputs"] else None)
            for s in schemas]
        conversations = [[{"role": "user", "content": p}] for p in prompts]
        outs = self.llm.chat(conversations, params, use_tqdm=True,
                             chat_template_kwargs={"enable_thinking": self.spec["enable_thinking"]})
        return [o.outputs[0].text for o in outs]


# 検索とリランクが終わったので、リランカーと BM25 を解放してから vLLM を起動する。
reranker = None
bm25 = None
free_gpu()
t0 = time.time()
generator = VLLMGenerator(GENERATOR)
print("[generator] %s (%.0f s)" % (GENERATOR["model"], time.time() - t0))


## Step 8: 共通入力から回答生成・提出ファイル保存

`generation_inputs` だけからプロンプトと出力 JSON スキーマを作り、回答を生成する。資料の追加取得や検索結果の再参照は行わない。
回答は提出形式で `submission.json` に保存する。実行のたびに最初から上書きし、採点は行わない。
`GEN_BATCH_SIZE` 問を生成するたびに保存するので、途中で中断しても生成済みの回答は残る。

In [ ]:
# [step8.1 generation] 編集不要
SUBMISSION_PATH = os.path.join(RUN_DIR, "submission.json")
# 1バッチ生成するごとに、それまでの回答で提出ファイルを上書きする。中断しても完了分が残る。
submission = []
write_json(SUBMISSION_PATH, submission)   # 前回の実行結果が残らないように最初に空にする
for start in range(0, len(generation_inputs), GEN_BATCH_SIZE):
    batch = generation_inputs[start:start + GEN_BATCH_SIZE]
    prompts = [answer_prompt(item["question"], item["contexts"]) for item in batch]
    outputs = generator.generate(prompts, [answer_schema(item["contexts"]) for item in batch])
    if len(outputs) != len(batch):
        raise ValueError("生成件数が質問数と一致しない")
    for item, raw in zip(batch, outputs):
        answer, cid = parse_answer(raw)
        submission.append({"q_id": item["q_id"], "answer": answer, "chunk": cid})
    write_json(SUBMISSION_PATH, submission)
    print("[generation] %d / %d 問を保存した" % (len(submission), len(generation_inputs)))
print("提出ファイル:", SUBMISSION_PATH)


In [ ]:
# 【件数確認セル】submission.json の中身（質問・回答）は表示しない。件数と文字数の集計だけを出す
import glob, json, os
p = globals().get("SUBMISSION_PATH")
if not p or not os.path.exists(str(p)):
    cands = sorted(set(glob.glob("/content/**/submission*.json", recursive=True) + glob.glob("/content/drive/MyDrive/**/submission*.json", recursive=True)), key=os.path.getmtime)
    print("SUBMISSION_PATH が使えないので、新しい順に探した（先頭が最新）:", cands[::-1][:3])
    p = cands[-1] if cands else None
if not p:
    raise SystemExit("submission.json が見つからない")
print("対象:", p)
s = json.load(open(p, encoding="utf-8"))
if not isinstance(s, list) or not s or not isinstance(s[0], dict):
    raise SystemExit("想定外の形式: %s" % type(s).__name__)
print("1件のキー:", sorted(s[0].keys()))
ans = [(x.get("answer") or "") for x in s]
lens = sorted(len(a) for a in ans)
print("件数:", len(s), "/ chunk=None:", sum(1 for x in s if x.get("chunk") is None), "/ answer が空:", sum(1 for a in ans if not a),
      "/ 20文字超:", sum(1 for n in lens if n > 20), "/ 最大文字数:", lens[-1], "/ 中央値:", lens[len(lens) // 2])